# Atividade: construindo regras para classificar frutas

Você receberá exemplos de maçãs, tangerinas, laranjas e limões. Observe os dados e crie suas próprias regras com `if`, `elif` e `else`.

Execute as células na ordem. Não procure o gabarito do teste antes de congelar suas regras. Ao longo do notebook, usaremos desde já nomes comuns em aprendizado de máquina — como `train`, `test`, `X`, `y`, `fit` e `predict` — e construiremos o significado de cada um.

## 1. Abrir os exemplos conhecidos

O código abaixo lê um arquivo CSV com Pandas. Cada linha representa uma fruta. As colunas `mass`, `width`, `height` e `color_score` descrevem a fruta; `fruit_name` informa seu nome. Nos dados originais, tangerina aparece como `mandarin`.

In [ ]:
import pandas as pd

TRAIN_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/train.csv'
train = pd.read_csv(TRAIN_URL)

FEATURES = ['mass', 'width', 'height', 'color_score']
TARGET = 'fruit_name'

X_train = train[FEATURES]
y_train = train[TARGET]

train.head()

In [ ]:
print(f'Rows: {len(train)}')
print('Columns:', list(train.columns))
y_train.value_counts()

Antes de programar, observe algumas linhas de cada fruta. Há a mesma quantidade de exemplos para todas as classes? Que consequência isso pode ter?

In [ ]:
train.groupby(TARGET)[FEATURES].agg(['min', 'max'])

Responda antes de continuar:

1. Qual fruta parece mais fácil de reconhecer? Por quê?
2. Quais duas frutas mais se confundem?
3. Uma característica sozinha parece suficiente para todas as decisões?
4. Esboce no papel uma primeira sequência de perguntas do tipo: se isto acontecer, então...

## 2. Escrever as regras

Em aprendizado de máquina, costumamos chamar as entradas de `X` e as respostas conhecidas de `y`. Por isso, separamos acima `X_train` e `y_train`: são as entradas e respostas usadas durante o ajuste.

Mais adiante, uma biblioteca poderá aprender automaticamente com uma chamada como `model.fit(X_train, y_train)` e gerar respostas com `model.predict(X_test)`. Nesta atividade, você fará o papel do `fit` manualmente: observará `X_train` e `y_train` e ajustará as condições da função `predict_fruit`.

Complete a função abaixo. Ela deve devolver exatamente um destes nomes: `apple`, `mandarin`, `orange` ou `lemon`. Você pode criar quantas condições forem necessárias, mas deve conseguir explicar cada uma delas.

In [ ]:
def predict_fruit(mass, width, height, color_score):
    # Apague a linha abaixo e escreva suas regras com if, elif e else.
    return 'apple'

# Teste manualmente uma fruta antes de aplicar a função a todas as linhas.
predict_fruit(mass=192, width=8.4, height=7.3, color_score=0.55)

In [ ]:
def predict(X):
    return X.apply(
        lambda row: predict_fruit(
            row['mass'],
            row['width'],
            row['height'],
            row['color_score'],
        ),
        axis=1,
    )

train_results = train[['id'] + FEATURES].copy()
train_results['y_true'] = y_train
train_results['y_pred'] = predict(X_train)
train_results['correct'] = train_results['y_pred'] == train_results['y_true']
train_results.head()

In [ ]:
train_accuracy = train_results['correct'].mean()
print(f'Train accuracy: {train_accuracy:.1%}')
display(train_results.loc[~train_results['correct']])

Ajuste suas regras olhando apenas para esses exemplos. Depois responda:

- Que mudanças aumentaram os acertos?
- Quais erros foram difíceis de corrigir sem criar outro erro?
- Sua função ficou simples de explicar?
- Uma acurácia alta aqui garante que a função acertará frutas novas?

## 3. Congelar as regras

Antes de abrir os exemplos novos:

1. anote sua acurácia nos exemplos conhecidos;
2. estime quantas das 11 frutas novas sua função acertará;
3. registre uma justificativa curta;
4. salve esta versão do notebook e não altere mais a função até avaliar o teste.

**Minha estimativa:** ... acertos em 11.

**Minha justificativa:** ...

## 4. Fazer previsões para frutas novas

O próximo arquivo não contém `fruit_name`. Gere as previsões sem alterar as regras.

In [ ]:
TEST_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/test.csv'
test = pd.read_csv(TEST_URL)
X_test = test[FEATURES]
test

In [ ]:
test_predictions = test[['id']].copy()
test_predictions['prediction'] = predict(X_test)
test_predictions.to_csv('fruit_predictions.csv', index=False)
display(test_predictions)

try:
    from google.colab import files
    files.download('fruit_predictions.csv')
except ImportError:
    print('The file fruit_predictions.csv was saved in the current folder.')

## 5. O que você acabou de fazer?

Agora podemos dar nome às partes da experiência:

- massa, largura, altura e medida de cor são as **características** (*features*);
- o nome da fruta é a **classe** ou **rótulo** (*label*);
- `X` reúne as **características** (*features*) usadas como entrada;
- `y` reúne a **classe** ou **rótulo** (*label*) que queremos prever;
- sua função de regras é um **modelo**: `predict(X)` transforma características em previsões;
- `X_train` e `y_train` formam os **dados de treino**;
- você fez o papel de `fit` manualmente, ajustando o modelo com esses dados;
- `X_test` contém os **dados de teste**, que o modelo ainda não tinha visto;
- funcionar em exemplos novos é o que chamamos de **generalização**.

Você ainda não treinou automaticamente um algoritmo de aprendizado de máquina. A atividade separou as etapas para que fosse possível sentir a dificuldade de formular regras e avaliar se elas generalizam.

## 6. Avaliar depois da liberação do gabarito

Cole abaixo o link direto fornecido pelo professor. Execute esta seção somente depois de entregar ou congelar suas previsões.

In [ ]:
ANSWER_KEY_URL = ''  # Cole aqui o link direto liberado pelo professor.

if not ANSWER_KEY_URL.strip():
    print('O gabarito ainda não foi informado. Preserve suas previsões e volte depois.')
else:
    answer_key = pd.read_csv(ANSWER_KEY_URL)
    test_results = test_predictions.merge(answer_key, on='id', validate='one_to_one')
    test_results = test_results.rename(
        columns={'fruit_name': 'y_true', 'prediction': 'y_pred'}
    )
    test_results['correct'] = test_results['y_pred'] == test_results['y_true']
    test_accuracy = test_results['correct'].mean()

    print(f'Test accuracy: {test_accuracy:.1%}')
    print('\nAcurácia por classe:')
    accuracy_by_class = test_results.groupby('y_true')['correct'].mean()
    display(accuracy_by_class.rename('accuracy'))

    print('Matriz de confusão:')
    confusion_matrix = pd.crosstab(
        test_results['y_true'],
        test_results['y_pred'],
        rownames=['y_true'],
        colnames=['y_pred'],
        dropna=False,
    )
    display(confusion_matrix)

    print('Erros:')
    display(test_results.loc[~test_results['correct']])

## 7. Fechamento

Responda com suas palavras:

1. O resultado do teste ficou acima ou abaixo de sua estimativa?
2. Compare a acurácia nos dados de treino com a acurácia no teste. O que mudou?
3. Qual classe teve o pior resultado? A acurácia total deixava isso evidente?
4. Observe os erros. Há frutas com valores fora dos intervalos vistos no treino?
5. O que você mudaria nas regras? Por que não devemos usar o mesmo teste repetidamente para fazer esses ajustes?
6. Que vantagem um algoritmo de treinamento poderia ter sobre o ajuste manual? Que cuidado ainda continuaria necessário?